In [12]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [13]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [14]:
def search(query):
    results = index.search(query=query, num_results=5)
    return results

def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [30]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the documentation database for relevant results based on a query string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up in the index"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "add_entry",
            "description": "Add a new documentation entry to the index.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The source filename"},
                    "title": {"type": "string", "description": "The title of the entry"},
                    "description": {"type": "string", "description": "A short description"},
                    "content": {"type": "string", "description": "The full content"}
                },
                "required": ["filename", "title", "description", "content"],
                "additionalProperties": False
            }
        }
    }
]


In [44]:
def make_call(tool_call):
    """Execute the function based on tool call"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        
        if name == 'search':
            # Replace with your real search
            result = index.search(**arguments)  # Your actual search function
        elif name == 'add_entry':
            # Replace with your real add_entry
            result = add_entry(**arguments)  # Your actual add_entry function
        else: 
            result = f'Unknown tool: "{name}"'
        
        return {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result),
        }
    except Exception as e:
        print(f"Error in make_call: {e}")
        return {
            "role": "tool", 
            "tool_call_id": tool_call.id,
            "content": f"Error: {str(e)}",
        }

In [50]:
instructions = """You are a documentation assistant. Follow this workflow EXACTLY:

CRITICAL: Keep your explanations BRIEF (2-3 sentences maximum). Do NOT include code in explanations.

When searching:
1. Provide a SHORT explanation (max 3 sentences)
2. Call the search tool
3. After gathering results, provide final answer

Be concise. Avoid long responses until the final answer."""

In [33]:
test_messages = [
    {"role": "system", "content": "You are a helpful assistant with access to search and add_entry tools."},
    {"role": "user", "content": "How do I create a dashboard?"}
]

print("Testing API call...")
try:
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Try the 8B model instead
        messages=test_messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.7,
        max_tokens=100,  # Limit for testing
    )
    print("SUCCESS!")
    print(response.choices[0].message)
except Exception as e:
    print(f"FAILED: {e}")

Testing API call...
SUCCESS!
ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='djtcfrkhn', function=Function(arguments='{"query":"create dashboard"}', name='search'), type='function')])


In [ ]:
# Initialize conversation
message_history = [
    {"role": "system", "content": instructions},
]

question = "How do I create a dashboard in Evidently?"
message_history.append({"role": "user", "content": question})

max_iterations = 4
iteration_count = 0

while iteration_count < max_iterations:
    print(f"\n--- Iteration {iteration_count + 1} ---")
    
    # Call Groq with tools
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Using the working model
        messages=message_history,
        tools=tools,
        tool_choice="auto",
        temperature=0.7,
    )
    
    assistant_message = response.choices[0].message
    
    def truncate_message(content, max_length=2000):
        """Truncate long messages to prevent context overflow"""
        if content and len(content) > max_length:
            return content[:max_length] + "... [truncated]"
        return content

    # Create clean message for history (without annotations)
    clean_message = {
        "role": assistant_message.role,
        "content": truncate_message(assistant_message.content, max_length=1500),
    }
    if assistant_message.tool_calls:
        clean_message["tool_calls"] = assistant_message.tool_calls
    
    message_history.append(clean_message)
    
    # Check for tool calls
    if assistant_message.tool_calls:
            # Before executing, check if the assistant also provided text
        print(f"💭 Explanation: {assistant_message.content}")
        print(f"🤖 Model requested: {len(assistant_message.tool_calls)} tool call(s)")
        
        # Execute each tool call
        for tool_call in assistant_message.tool_calls:
            print(f"📞 Executing {tool_call.function.name}...")
            tool_result = make_call(tool_call)
            message_history.append(tool_result)
            print(f"✅ Completed {tool_call.function.name}")
        
    elif assistant_message.content:
        print("\n" + "="*50)
        print("ASSISTANT FINAL ANSWER:")
        print("="*50)
        print(assistant_message.content)
        break
    
    iteration_count += 1

if iteration_count >= max_iterations:
    print("Reached max iterations without final answer")


--- Iteration 1 ---
💭 Explanation: To create a dashboard in Evidently, you would typically start by defining the dashboard, specifying the metrics and visualizations that you want to include. 


🤖 Model requested: 1 tool call(s)
📞 Executing search...
✅ Completed search

--- Iteration 2 ---


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kmhb4tp4fv39tkevk5cd9r3v` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7285, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [35]:
# Test with versatile model (fails)
try:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Search for dashboard"}],
        tools=tools,
    )
    print("Versatile works?")
except Exception as e:
    print(f"Versatile fails: {e}")

# Test with instant model (works)
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Search for dashboard"}],
    tools=tools,
)
print("Instant works!")

Versatile fails: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search{"query": "dashboard"}</function>'}}
Instant works!


In [55]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'whisper-large-v3', 'object': 'model', 'created': 1693721698, 'owned_by': 'OpenAI', 'active': True, 'context_window': 448, 'public_apps': None, 'max_completion_tokens': 448}, {'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'meta-llama/llama-4-scout-17b-16e-instruct', 'object': 'model', 'created': 1743874824, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'qwen/qwen3-32b', 'object': 'model', 'created': 1748396646, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 40

In [63]:
print("📚 Evidently Documentation Assistant")
print("Type 'stop' to exit, 'reset' to clear conversation")
print("-" * 50)

message_history = [
    {"role": "system", "content": instructions},
]

while True:
    user_input = input("\nYou: ").strip()
    
    if user_input.lower() == 'stop':
        print("Goodbye!")
        break
    
    if user_input.lower() == 'reset':
        message_history = [{"role": "system", "content": instructions}]
        print("Conversation reset!")
        continue
    
    message_history.append({"role": "user", "content": user_input})
    
    iteration_count = 0
    max_iterations = 5
    
    while iteration_count < max_iterations:
        print(f"\n[Step {iteration_count + 1}]")
        
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=message_history,
            tools=tools,
            tool_choice="auto",
            temperature=0.7,
        )
        
        assistant_message = response.choices[0].message
        
        # Clean and append
        clean_message = {"role": assistant_message.role, "content": assistant_message.content}
        if assistant_message.tool_calls:
            clean_message["tool_calls"] = assistant_message.tool_calls
        
        message_history.append(clean_message)
        
        if assistant_message.tool_calls:
            print(f"🔧 Executing {len(assistant_message.tool_calls)} tool(s)...")
            for tool_call in assistant_message.tool_calls:
                tool_result = make_call(tool_call)
                message_history.append(tool_result)
                print(f"   ✓ {tool_call.function.name}")
        else:
            print(f"\n🤖 Assistant: {assistant_message.content}")
            break
        
        iteration_count += 1

📚 Evidently Documentation Assistant
Type 'stop' to exit, 'reset' to clear conversation
--------------------------------------------------

[Step 1]
🔧 Executing 3 tool(s)...
   ✓ search
   ✓ search
   ✓ search

[Step 2]


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kmhb4tp4fv39tkevk5cd9r3v` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 23928, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}